# Data Cleaning — custom_esci_with_user_query.csv

This notebook performs data cleaning on the custom ESCI dataset:
- Load the dataset (CSV or Excel, auto-detected)
- Drop junk columns
- Fix invalid labels (misaligned rows)
- Strip whitespace
- Remove repeated filler text from queries/titles
- Remove duplicates
- Validate label distribution
- Save cleaned dataset as CSV for training

In [14]:
import pandas as pd
import os
import re
from collections import Counter
from IPython.display import display

## 1. Load the Dataset

In [ ]:
DATASET_DIR = r"C:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci"
FILE_PATH = os.path.join(DATASET_DIR, "esci_100k_plus_dataset.csv")

# Auto-detect file format
if FILE_PATH.endswith('.csv'):
    df = pd.read_csv(FILE_PATH, encoding='latin-1')
else:
    df = pd.read_excel(FILE_PATH, engine='openpyxl')

print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

Original shape: (640000, 4)
Columns: ['query', 'product_title', 'label', 'user_query']


,query,product_title,label,user_query
0,rice,Rice,E,I am looking for rice for everyday cooking
1,rice,Sinandomeng Rice,E,I am looking for rice for everyday cooking
2,rice,Jasmine Rice,S,I am looking for rice for everyday cooking
3,rice,Brown Rice,S,I am looking for rice for everyday cooking
4,rice,Rice Cooker,C,I am looking for rice for everyday cooking
5,rice,Patis,C,I am looking for rice for everyday cooking
6,rice,Shampoo,I,I am looking for rice for everyday cooking
7,rice,Notebook,I,I am looking for rice for everyday cooking
8,bread,White Bread,E,I want bread for breakfast sandwiches
9,bread,Pandesal,E,I want bread for breakfast sandwiches


## 2. Drop Junk Columns

In [16]:
# Drop any 'Unnamed' columns (artifacts from Excel)
unnamed_cols = [col for col in df.columns if col.startswith('Unnamed')]
if unnamed_cols:
    print(f"Dropping junk columns: {unnamed_cols}")
    df = df.drop(columns=unnamed_cols)
else:
    print("No junk columns found.")

print(f"Shape after drop: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

No junk columns found.
Shape after drop: (640000, 4)
Columns: ['query', 'product_title', 'label', 'user_query']


## 3. Strip Whitespace

In [17]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Uppercase labels for consistency
df['label'] = df['label'].str.upper()

print("Whitespace stripped and labels uppercased.")

Whitespace stripped and labels uppercased.


## 3.5 Remove Repeated Filler Text from Queries

Some queries have repeated padding like `"Set Pack Set Pack Set Pack Set"` appended.
This step detects and removes those repeated trailing patterns.

In [18]:
def remove_repeated_filler(text):
    """
    Remove repetitive word patterns from text:
    1. Word-cycling: 'beef meat can food meal beef meat can meal...' → 'beef meat can food meal'
       Detected via low unique-word ratio; fixed by keeping first occurrence of each word.
    2. Trailing repeats: '... Set Pack Set Pack Set Pack Set' → strips the repeated tail.
    """
    if not isinstance(text, str):
        return text

    words = text.split()
    if len(words) < 6:
        return text

    # --- Method 1: Word-cycling (low unique word ratio) ---
    unique_ratio = len(set(w.lower() for w in words)) / len(words)
    if unique_ratio <= 0.6:
        seen = set()
        result = []
        for w in words:
            wl = w.lower()
            if wl not in seen:
                seen.add(wl)
                result.append(w)
        return ' '.join(result)

    # --- Method 2: Trailing repeated patterns ---
    for pattern_len in range(1, 5):
        if len(words) < pattern_len * 3:
            continue

        candidate = words[-pattern_len:]

        repeat_count = 0
        pos = len(words)
        while pos >= pattern_len:
            chunk = words[pos - pattern_len : pos]
            if chunk == candidate:
                repeat_count += 1
                pos -= pattern_len
            else:
                break

        if repeat_count >= 3:
            clean_words = words[:pos + pattern_len]
            return ' '.join(clean_words)

    return text

# Save originals before cleaning so we can count changes without re-reading the file
original_queries = df['query'].copy()

# Apply to both columns
df['query'] = df['query'].apply(remove_repeated_filler)
df['product_title'] = df['product_title'].apply(remove_repeated_filler)

# Count changed rows (fillna handles NaN safely)
query_changed = (original_queries.fillna('') != df['query'].fillna('')).sum()
print(f"Queries cleaned (filler removed): {query_changed} out of {len(df)}")
print("Filler removal also applied to product_title column.")

Queries cleaned (filler removed): 0 out of 640000
Filler removal also applied to product_title column.


## 4. Check for Invalid Labels

In [19]:
print(f"Current shape after filler removal: {df.shape}")
df.head(3)

Current shape after filler removal: (640000, 4)


,query,product_title,label,user_query
0,rice,Rice,E,I am looking for rice for everyday cooking
1,rice,Sinandomeng Rice,E,I am looking for rice for everyday cooking
2,rice,Jasmine Rice,S,I am looking for rice for everyday cooking


In [20]:
valid_labels = {'E', 'S', 'C', 'I'}
invalid_mask = ~df['label'].isin(valid_labels)
invalid_count = invalid_mask.sum()

print(f"Invalid label rows: {invalid_count}")

if invalid_count > 0:
    # Some rows have misaligned columns: description landed in 'label', actual label in 'user_query'
    # Fix: where label is invalid but user_query IS a valid label, swap them
    fixable_mask = invalid_mask & df['user_query'].isin(valid_labels)
    fixable_count = fixable_mask.sum()
    print(f"  Fixable (label in user_query column): {fixable_count}")

    if fixable_count > 0:
        df.loc[fixable_mask, 'label'] = df.loc[fixable_mask, 'user_query']
        df.loc[fixable_mask, 'user_query'] = pd.NA
        print(f"  Realigned {fixable_count} rows.")

    # Re-check for remaining invalid labels (e.g. header rows, Excel artifacts)
    still_invalid = ~df['label'].isin(valid_labels)
    remaining = still_invalid.sum()
    if remaining > 0:
        print(f"\n  Remaining unfixable rows: {remaining}")
        display(df[still_invalid].head(10))
        df = df[~still_invalid].copy()
        print(f"  Dropped {remaining} unfixable rows.")

    print(f"\nShape after label fix: {df.shape}")
else:
    print("All labels are valid.")

Invalid label rows: 0
All labels are valid.


## 5. Check for Nulls

In [21]:
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)

if null_counts.sum() > 0:
    print(f"\nDropping {null_counts.sum()} rows with nulls...")
    df = df.dropna(subset=['query', 'product_title', 'label'])
    print(f"Shape after dropping nulls: {df.shape}")
else:
    print("No nulls found.")

Null counts per column:
query            0
product_title    0
label            0
user_query       0
dtype: int64
No nulls found.


## 6. Remove Duplicates

In [22]:
num_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {num_duplicates} duplicates.")
    print(f"Shape after dedup: {df.shape}")
else:
    print("No duplicates found.")

Duplicate rows: 639920
Removed 639920 duplicates.
Shape after dedup: (80, 4)


## 7. Dataset Summary

In [23]:
print(f"Final shape: {df.shape}")
print(f"Unique queries: {df['query'].nunique()}")
print(f"Unique products: {df['product_title'].nunique()}")
print(f"\nLabel distribution:")

label_names = {'E': 'Exact', 'S': 'Substitute', 'C': 'Complement', 'I': 'Irrelevant'}
label_counts = df['label'].value_counts()
for label in ['E', 'S', 'C', 'I']:
    count = label_counts.get(label, 0)
    pct = count / len(df) * 100
    print(f"  {label} ({label_names[label]}): {count} ({pct:.1f}%)")

Final shape: (80, 4)
Unique queries: 10
Unique products: 62

Label distribution:
  E (Exact): 20 (25.0%)
  S (Substitute): 20 (25.0%)
  C (Complement): 20 (25.0%)
  I (Irrelevant): 20 (25.0%)


In [24]:
# Show sample rows per label
for label in ['E', 'S', 'C', 'I']:
    print(f"\n--- {label} ({label_names[label]}) sample ---")
    display(df[df['label'] == label].sample(min(5, len(df[df['label'] == label])), random_state=42))


--- E (Exact) sample ---


,query,product_title,label,user_query
0,rice,Rice,E,I am looking for rice for everyday cooking
65,cooking oil,Coconut Oil,E,I need cooking oil for frying
57,soy sauce,Silver Swan Soy Sauce,E,I need soy sauce for cooking
1,rice,Sinandomeng Rice,E,I am looking for rice for everyday cooking
32,eggs,Chicken Eggs,E,I want eggs for cooking breakfast



--- S (Substitute) sample ---


,query,product_title,label,user_query
2,rice,Jasmine Rice,S,I am looking for rice for everyday cooking
67,cooking oil,Corn Oil,S,I need cooking oil for frying
59,soy sauce,Tamari,S,I need soy sauce for cooking
3,rice,Brown Rice,S,I am looking for rice for everyday cooking
34,eggs,Quail Eggs,S,I want eggs for cooking breakfast



--- C (Complement) sample ---


,query,product_title,label,user_query
4,rice,Rice Cooker,C,I am looking for rice for everyday cooking
69,cooking oil,Garlic,C,I need cooking oil for frying
61,soy sauce,Garlic,C,I need soy sauce for cooking
5,rice,Patis,C,I am looking for rice for everyday cooking
36,eggs,Cooking Oil,C,I want eggs for cooking breakfast



--- I (Irrelevant) sample ---


,query,product_title,label,user_query
6,rice,Shampoo,I,I am looking for rice for everyday cooking
71,cooking oil,Shampoo,I,I need cooking oil for frying
63,soy sauce,Shampoo,I,I need soy sauce for cooking
7,rice,Notebook,I,I am looking for rice for everyday cooking
38,eggs,Shampoo,I,I want eggs for cooking breakfast


## 8. Save Cleaned Dataset

In [25]:
# Save cleaned CSV
CSV_OUTPUT = os.path.join(DATASET_DIR, "custom_esci_template.csv")
df.to_csv(CSV_OUTPUT, index=False)
print(f"Saved cleaned CSV to: {CSV_OUTPUT}")
print(f"Rows: {len(df)}")

Saved cleaned CSV to: C:\Moi\Thesis\Code\RetailTalkFolder\classification_identification\custom_esci\custom_esci_template.csv
Rows: 80


In [26]:
# Verify saved CSV
df_verify = pd.read_csv(CSV_OUTPUT)
print(f"Verified CSV shape: {df_verify.shape}")
print(f"Remaining duplicates: {df_verify.duplicated().sum()}")
print(f"Invalid labels: {(~df_verify['label'].isin(valid_labels)).sum()}")

Verified CSV shape: (80, 4)
Remaining duplicates: 0
Invalid labels: 0
